# <center> <img src="../img/ITESOLogo.png" alt="ITESO" width="480" height="130"> </center>
# <center> **Departamento de Electrónica, Sistemas e Informática** </center>
---
## <center> **Big Data** </center>
---
### <center> **Spring 2026** </center>
---
### <center> **Examples on Machine Learning: K-means** </center>
---
**Profesor**: Pablo Camarillo Ramirez

# Create SparkSession

In [14]:
from SparkUtils import SparkUtils

from pyspark.ml.feature import VectorAssembler
from pyspark.ml.clustering import KMeans
from pyspark.ml.clustering import KMeansModel
from pyspark.ml.evaluation import ClusteringEvaluator

In [2]:
MASTER_URL = "spark://spark-master:7077"
APP_NAME = "ML: K-Means"

spark = SparkUtils(MASTER_URL, APP_NAME)._spark

spark

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/28 00:38:26 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


# Example 1: Clustering with 2D points

In [3]:
# Sample data in Python (e.g., 2D points)
data = [
    (0, 1.0, 1.0),
    (1, 2.0, 1.0),
    (2, 4.0, 5.0),
    (3, 5.0, 5.0),
    (4, 10.0, 10.0),
    (5, 12.0, 11.0)
]

# Define schema for the DataFrame
schema = SparkUtils.generate_schema([("id", "int"), ("x", "float"), ("y", "float")])

# Create DataFrame for k means
random_points_df = spark.createDataFrame(data, schema)

## Assemble the features into a single vector column

In [5]:
assembler = VectorAssembler(inputCols=["x", "y"], outputCol="features")

assembled_df = assembler.transform(random_points_df)

## Configure K-means

In [7]:
kmeans = KMeans().setK(3).setSeed(73)

## Train model

In [8]:
model = kmeans.fit(assembled_df)

print("K-means model trained successfully")

kmeans_model_path = "/opt/spark/work-dir/data/mlmodels/kmeans/2D"
model.write().overwrite().save(kmeans_model_path)
model.__class__

26/04/28 00:39:59 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS


K-means model trained successfully


pyspark.ml.clustering.KMeansModel

## Get Predictions

In [12]:
k_model = KMeansModel.load(kmeans_model_path)

predictions = k_model.transform(assembled_df)

predictions.show()

+---+----+----+-----------+----------+
| id|   x|   y|   features|prediction|
+---+----+----+-----------+----------+
|  0| 1.0| 1.0|  [1.0,1.0]|         2|
|  1| 2.0| 1.0|  [2.0,1.0]|         2|
|  2| 4.0| 5.0|  [4.0,5.0]|         0|
|  3| 5.0| 5.0|  [5.0,5.0]|         0|
|  4|10.0|10.0|[10.0,10.0]|         1|
|  5|12.0|11.0|[12.0,11.0]|         1|
+---+----+----+-----------+----------+



## Evaluate model

In [15]:
# Evaluate clustering by computing Silhouette score
evaluator = ClusteringEvaluator()
silhouette = evaluator.evaluate(predictions)
print(f"Silhouette score: {silhouette}")

# Show the result
print("Cluster Centers: ")
for center in model.clusterCenters():
    print(center)

Silhouette score: 0.9494652547284126
Cluster Centers: 
[4.5 5. ]
[11.  10.5]
[1.5 1. ]


# Lab 13: Clustering Wine dataset with K-means

In [18]:
# Downlod dataset from https://www.kaggle.com/datasets/harrywang/wine-dataset-for-clustering

columns_types = [
    ("Alcohol", "float"),
    ("Malic_Acid", "float"),
    ("Ash", "float"),
    ("Ash_Alcanity", "float"),
    ("Magnesium", "float"),
    ("Total_Phenols", "float"),
    ("Flavanoids", "float"),
    ("Nonflavanoid_Phenols", "float"),
    ("Proanthocyanins", "float"),
    ("Color_Intensity", "float"),
    ("Hue", "float"),
    ("OD280", "float"),
    ("Proline", "float")
]

# Define schema for the DataFrame
wines_schema = SparkUtils.generate_schema(columns_types)

# Create DataFrame from wines csv
wines_df = spark \
    .read \
    .option("header", "true") \
    .schema(wines_schema) \
    .csv("/opt/spark/work-dir/data/ml/kmeans")

assembler = VectorAssembler(inputCols=[x for x,_ in columns_types], outputCol="features")
assembled_df = assembler.transform(wines_df)

# TODO: Find the optimial K
# TODO: Add the code here to iterate from k = 2, 4, .., 10 and get the silhouette score for each k

k = 2

for k in range(2, 11, 2):
    kmeans = KMeans().setK(k).setSeed(13)

    model = kmeans.fit(assembled_df)

    print(f"K-means model trained successfully for {k} clusters")

    # Evaluate clustering by computing Silhouette score
    evaluator = ClusteringEvaluator()
    predictions = model.transform(assembled_df)
    silhouette = evaluator.evaluate(predictions)
    print(f"Silhouette score: {silhouette}")

K-means model trained successfully for 2 clusters
Silhouette score: 0.8193526758797327
K-means model trained successfully for 4 clusters
Silhouette score: 0.730306656362107
K-means model trained successfully for 6 clusters
Silhouette score: 0.7358902017475427
K-means model trained successfully for 8 clusters
Silhouette score: 0.6986940310339242
K-means model trained successfully for 10 clusters
Silhouette score: 0.6914941969954018


The best silhouette score is 0.81 with `k = 2`. Given that it is the one that is closer to `1`, it means that points were mostly clustered correctly.

In [19]:
spark.stop()